In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS hvacapp_dev3.refined;

In [0]:
%sql
CREATE OR REPLACE TABLE hvacapp_dev3.refined.silver_telemetry_wide (
  event_ts TIMESTAMP,
  site_id STRING,
  building_id STRING,
  equipment_id STRING,

  chw_supply_temp_c DOUBLE,
  chw_return_temp_c DOUBLE,
  hvac_power_kw DOUBLE,
  chw_flow_lpm DOUBLE,
  pump_speed_pct DOUBLE,

  quality_status STRING,
  created_ts TIMESTAMP
)
USING DELTA;

In [0]:
%sql
-- We are converting:
-- From (Bronze) 1 row = 1 metric
-- To (Silver) 1 row = ALL metrics for that timestamp + equipment
-- This is called pivoting
--      Group by timestamp + equipment
--      Convert rows → columns
--      Combine all sensor readings into one row

SELECT
  event_ts,
  site_id,
  building_id,
  equipment_id,

  MAX(CASE WHEN metric_name = 'chw_supply_temp_c' THEN metric_value END) AS chw_supply_temp_c,
  MAX(CASE WHEN metric_name = 'chw_return_temp_c' THEN metric_value END) AS chw_return_temp_c,
  MAX(CASE WHEN metric_name = 'hvac_power_kw' THEN metric_value END) AS hvac_power_kw,
  MAX(CASE WHEN metric_name = 'chw_flow_lpm' THEN metric_value END) AS chw_flow_lpm,
  MAX(CASE WHEN metric_name = 'pump_speed_pct' THEN metric_value END) AS pump_speed_pct,

  'GOOD' AS quality_status,
  current_timestamp() AS created_ts

FROM hvacapp_dev3.raw.bronze_hvac_telemetry

GROUP BY
  event_ts,
  site_id,
  building_id,
  equipment_id

ORDER BY event_ts DESC;

In [0]:
%sql
-- wrap with INSERT:

INSERT INTO hvacapp_dev3.refined.silver_telemetry_wide
SELECT
  date_trunc('minute', event_ts) AS event_ts,
  site_id,
  building_id,
  'HVAC_SYSTEM_001' AS equipment_id,

  MAX(CASE WHEN metric_name = 'chw_supply_temp_c' THEN metric_value END) AS chw_supply_temp_c,
  MAX(CASE WHEN metric_name = 'chw_return_temp_c' THEN metric_value END) AS chw_return_temp_c,
  MAX(CASE WHEN metric_name = 'hvac_power_kw' THEN metric_value END) AS hvac_power_kw,
  MAX(CASE WHEN metric_name = 'chw_flow_lpm' THEN metric_value END) AS chw_flow_lpm,
  MAX(CASE WHEN metric_name = 'pump_speed_pct' THEN metric_value END) AS pump_speed_pct,

  'GOOD' AS quality_status,
  current_timestamp() AS created_ts

FROM hvacapp_dev3.raw.bronze_hvac_telemetry
GROUP BY
  date_trunc('minute', event_ts),
  site_id,
  building_id;

-- DELETE FROM hvacapp_dev3.refined.silver_telemetry_wide;

-- SELECT COUNT(*) AS bronze_count
-- FROM hvacapp_dev3.raw.bronze_hvac_telemetry;

-- SHOW TABLES IN hvacapp_dev3.refined;

-- DESCRIBE TABLE hvacapp_dev3.refined.silver_telemetry_wide;


In [0]:
%sql
SELECT *
FROM hvacapp_dev3.refined.silver_telemetry_wide
ORDER BY event_ts DESC;

In [0]:
%sql
-- # Adding validation silver_telemetry_clean for trustworthiness -- remove bad data
-- In real HVAC system:
    -- Wide might contain:
    -- sensor failure (0 flow)
    -- spike (temp = 100°C)
    -- negative power
    -- missing values

CREATE OR REPLACE VIEW hvacapp_dev3.refined.silver_telemetry_clean AS
SELECT *
FROM hvacapp_dev3.refined.silver_telemetry_wide
WHERE
  chw_supply_temp_c BETWEEN 3.00 AND 15.00
  AND chw_return_temp_c BETWEEN 5.00 AND 25.00
  AND hvac_power_kw > 0
  AND chw_flow_lpm > 0
  AND pump_speed_pct > 0;


In [0]:
%sql
-- Check clean data
SELECT *
FROM hvacapp_dev3.refined.silver_telemetry_clean
ORDER BY event_ts DESC;

In [0]:
%sql
-- check all metrics present
SELECT
  COUNT(*) AS total_rows,
  COUNT(chw_supply_temp_c) AS supply_count,
  COUNT(chw_return_temp_c) AS return_count,
  COUNT(hvac_power_kw) AS power_count,
  COUNT(chw_flow_lpm) AS flow_count,
  COUNT(pump_speed_pct) AS pump_speed
FROM hvacapp_dev3.refined.silver_telemetry_wide;

In [0]:
%sql
SELECT
  date_trunc('minute', event_ts) AS event_ts,
  site_id,
  building_id,
  equipment_id,
  MAX(CASE WHEN metric_name = 'chw_supply_temp_c' THEN metric_value END) AS chw_supply_temp_c,
  MAX(CASE WHEN metric_name = 'chw_return_temp_c' THEN metric_value END) AS chw_return_temp_c,
  MAX(CASE WHEN metric_name = 'hvac_power_kw' THEN metric_value END) AS hvac_power_kw,
  MAX(CASE WHEN metric_name = 'chw_flow_lpm' THEN metric_value END) AS chw_flow_lpm,
  MAX(CASE WHEN metric_name = 'pump_speed_pct' THEN metric_value END) AS pump_speed_pct
FROM hvacapp_dev3.raw.bronze_hvac_telemetry
GROUP BY
  date_trunc('minute', event_ts),
  site_id,
  building_id,
  equipment_id;

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN chw_supply_temp_c BETWEEN 3.00 AND 15.00 THEN 1 ELSE 0 END) AS pass_supply,
  SUM(CASE WHEN chw_return_temp_c BETWEEN 5.00 AND 25.00 THEN 1 ELSE 0 END) AS pass_return,
  SUM(CASE WHEN hvac_power_kw > 0 THEN 1 ELSE 0 END) AS pass_power,
  SUM(CASE WHEN chw_flow_lpm > 0 THEN 1 ELSE 0 END) AS pass_flow
FROM hvacapp_dev3.refined.silver_telemetry_wide;